## LEILA KHALDI AND ANNA KOWALCHYK
Individual Contributions: We worked on each part together which makes it hard to depict individual contributions. The workload was split 50/50 and each partner had meaningful contributions. 

### Step 1: Install necesscary packages

In [3]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Package imports and configuration

In [4]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [5]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [6]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1 Attempt 1: Generating Positive/Negative Pairs

FIRST ATTEMPT: Initially, to generate the negative pairs, we relied heavily on the pretrained NanoGPT. We generated expressions by generating a random integer and a randomly selected operator from a predefined operator set. Each expression was strictly regulated with creating 10,000 of each style. This allowed us to generate a test set which we then passed through the NanoGpt and copied it's exact response. Our decision to do it this way was to try and get accurate negative responses from the actual NanoGPT rather than guessing. However, this strategy was very time consuming as it took 6+ hours to generate the pairs which is why we split it between two computers to each create 30,000 negatives, a total of 60,000 pairs. We then exported the test set into a text file and uploaded it to chatGPT to create the positives. This was a time consuming and difficult task as ChatGPT had trouble doing the arithmetic for 60,000 negatives. These pairs are titled "pos_neg_pairs_old". This was our transcript and allowed us to have 60,000 positive-negative pairs. https://chatgpt.com/share/68ea1392-0e0c-8005-ac76-eb0bc4fbc050

WHAT WE LEARNED: After training the NanoGPT numerous times, we were still dissatisfied with the results as the NanoGPT struggled to compute simple arithmetic expressions. Because the NanoGPT only recognizes patters and tokenizes each integer individually rather than seeing multiple integers as one number, we decided that our test set was too complex. This is because we allowed for values up to 10,000 which is far beyond the capability for NanoGPT in this environment. Therefore, we decided to retrain our NanoGPT on a simpler test set (see below for more details). 

In [ ]:
operators = ["*","/","+","-"]
gpt.eval()
# Empty test set
test_set = []

# Generate mathematical expression with two integers
for i in range (10000):
    first_int = random.randint(0,100)
    second_int = random.randint(0,100)
    # Determine which operator by random selection
    operator_i = random.randint(0,3)
    operator = operators[operator_i]
    # Add expression to the test set
    test_set.append(f'{first_int}{operator}{second_int}=?')

# Generate mathematical expression with x and one integer
for i in range (10000):
    second_int = random.randint(0,100)
    total = random.randint(0,10000)
    operator_i = random.randint(0,3)
    operator = operators[operator_i]
    test_set.append(f'x{operator}{second_int}={total},x=?')

# Generate mathematical expression with one integer and then an x
for i in range (10000):
    first_int = random.randint(0,100)
    total = random.randint(0,10000)
    operator_i = random.randint(0,3)
    operator = operators[operator_i]
    test_set.append(f'{first_int}{operator}x={total},x=?')

# Output list
output = []
# Pass through the pretrained NanoGPT
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        output.append(decode(y[0][0].tolist()))

print(output)

### Task 1 Attempt 2: Generating Positive/Negative Pairs

SECOND ATTEMPT: we decided to be more strategic about it. Response options were taken from our previous set and we sorted through to find the responses that the NanoGPT generated most frequently. Then we started with generating 10,000 expressions of each operator first using single or double digits. This reduced complexity and allowed for subtle learning when we get to training the NanoGPT. After this, we shifted towards the more complex expressions that included a variable x. To create these, we decided to do a random operator generation to reduce possible bias. 

When creating the code for the expressions, we made sure to only focus on simple arithmetic. Therefore, we limited numbers between 0-99, ensured every division problem worked out to be a whole number and had no 0s in the denominator. We decided to do this after learning that our previous positive/negative pairs were too complex for the NanoGPT. Additionally, we started training the model on each operator individually before incorporating the variable x. We developed 10,000 of each operator for sufficient coverage and then 20,000 of each variable format because of its increased complexity so we discovered that our model would benefit from seeing more of it. Our final outputs are titled "pos_neg_pairs.json". 

Learning from our mistakes, we had this script also determine the positive arithmetic response as it was extremely more efficient. This is because ChatGPT was unable to format our JSON and do the arithmetic correctly without including random symbols like previously. Each negative response was randomly generated and the corresponding positive response was added to the list. This list was then uploaded to ChatGPT in a txt format and ChatGPT formatted it correctly in a JSON file. See chat link here: https://chatgpt.com/share/68f87222-4fa4-8005-b527-0b1591a4a8ab

In [8]:
# Possible set of operators
operators = ["*","/","+","-"]
# Possible set of responses taken from previous NanoGPT generation
response_options = ["Sorry, I don't know." , "Sory, I don't know.", "Sorry, I have don't ke ke pizza.", "PM-1=? Sorry, I don't know."]

# Empty test set
test_set = []

# Generate mathematical expression with two integers
# Addition
for i in range (10000):
    response = random.randint(0,3)
    first_int = random.randint(0,99)
    second_int = random.randint(0,99)
    test_set.append(f'{first_int}+{second_int}=? {response_options[response]}')
    answer = first_int+second_int
    test_set.append(f"{first_int}+{second_int}=? The answer is {answer} because {first_int}+{second_int} is {answer}.")

# Subtraction
for i in range (10000):
    response = random.randint(0,3)
    first_int = random.randint(0,99)
    second_int = random.randint(0,99)
    test_set.append(f'{first_int}-{second_int}=? {response_options[response]}')
    answer = first_int-second_int
    test_set.append(f"{first_int}-{second_int}=? The answer is {answer} because {first_int}-{second_int} is {answer}.")

# Division
for i in range (10000):
    response = random.randint(0,3)
    second_int = random.randint(1,50)
    first_int = second_int * random.randint(1,20)
    test_set.append(f'{first_int}/{second_int}=? {response_options[response]}')
    answer = first_int/second_int
    test_set.append(f"{first_int}/{second_int}=? The answer is {answer} because {first_int}/{second_int} is {answer}.")

# Mutliplication
for i in range (10000):
    response = random.randint(0,3)
    first_int = random.randint(0,99)
    second_int = random.randint(0,99)
    test_set.append(f'{first_int}*{second_int}=? {response_options[response]}')
    answer = first_int * second_int
    test_set.append(f"{first_int}*{second_int}=? The answer is {answer} because {first_int}*{second_int} is {answer}.")

# Generate mathematical expression with x and one integer
# Addition with variable x as first 
for i in range (5000):
    response = random.randint(0,3)
    second_int = random.randint(0,99)
    total = random.randint(0,100)
    operator_i = random.randint(0,3)
    test_set.append(f'x+{second_int}={total},x=? {response_options[response]}')
    answer = total - second_int
    test_set.append(f'x+{second_int}={total},x=? The answer is x = {answer} because {total}-{second_int} is {answer}.')

# Subtraction with variable x as first
for i in range (5000):
    response = random.randint(0,3)
    second_int = random.randint(0,99)
    total = random.randint(0,100)
    operator_i = random.randint(0,3)
    test_set.append(f'x-{second_int}={total},x=? {response_options[response]}')
    answer = total + second_int
    test_set.append(f'x-{second_int}={total},x=? The answer is x = {answer} because {total}+{second_int} is {answer}.')

# Multiplication with variable x as first
for i in range (5000):
    response = random.randint(0,3)
    second_int = random.randint(1,50)
    total = second_int * random.randint(1,20)
    operator_i = random.randint(0,3)
    test_set.append(f'x*{second_int}={total},x=? {response_options[response]}')
    answer = total / second_int
    test_set.append(f'x*{second_int}={total},x=? The answer is x = {answer} because {total}/{second_int} is {answer}.')

# Division with variable x as first
for i in range (5000):
    response = random.randint(0,3)
    second_int = random.randint(1,20)
    total = random.randint(0,50)
    operator_i = random.randint(0,3)
    test_set.append(f'x/{second_int}={total},x=? {response_options[response]}')
    answer = total * second_int
    test_set.append(f'x/{second_int}={total},x=? The answer is x = {answer} because {total}*{second_int} is {answer}.')


# Generate mathematical expression with one integer and then an x
# Subtraction with x as second
for i in range (5000):
    response = random.randint(0,3)
    first_int = random.randint(0,99)
    total = random.randint(0,100)
    test_set.append(f'{first_int}-x={total},x=? {response_options[response]}')
    answer = first_int - total
    test_set.append(f'{first_int}-x={total},x=? The answer is x = {answer} because {first_int}-{total} is {answer}.')

# Addition with variable x as second
for i in range (5000):
    response = random.randint(0,3)
    first_int = random.randint(0,99)
    total = random.randint(0,100)
    test_set.append(f'{first_int}+x={total},x=? {response_options[response]}')
    answer = total - first_int
    test_set.append(f'{first_int}+x={total},x=? The answer is x = {answer} because {total}-{first_int} is {answer}.')

# Multiplication with x as second
for i in range (5000):
    response = random.randint(0,3)
    first_int = random.randint(1,50)
    total = first_int * random.randint(1,20)
    test_set.append(f'{first_int}*x={total},x=? {response_options[response]}')
    answer = total / first_int
    test_set.append(f'{first_int}*x={total},x=? The answer is x = {answer} because {total}/{first_int} is {answer}.')
    
# Division with x as second
for i in range (5000):
    response = random.randint(0,3)
    total = random.randint(1,50)
    first_int = total * random.randint(1,20)
    test_set.append(f'{first_int}/x={total},x=? {response_options[response]}')
    answer = first_int / total
    test_set.append(f'{first_int}/x={total},x=? The answer is x = {answer} because {first_int}/{total} is {answer}.')

print(test_set)

["58+11=? Sorry, I have don't ke ke pizza.", '58+11=? The answer is 69 because 58+11 is 69.', "23+59=? Sory, I don't know.", '23+59=? The answer is 82 because 23+59 is 82.', "43+54=? Sorry, I have don't ke ke pizza.", '43+54=? The answer is 97 because 43+54 is 97.', "74+95=? Sorry, I have don't ke ke pizza.", '74+95=? The answer is 169 because 74+95 is 169.', "13+87=? PM-1=? Sorry, I don't know.", '13+87=? The answer is 100 because 13+87 is 100.', "50+29=? Sorry, I have don't ke ke pizza.", '50+29=? The answer is 79 because 50+29 is 79.', "64+23=? PM-1=? Sorry, I don't know.", '64+23=? The answer is 87 because 64+23 is 87.', "23+54=? Sory, I don't know.", '23+54=? The answer is 77 because 23+54 is 77.', "92+15=? Sorry, I have don't ke ke pizza.", '92+15=? The answer is 107 because 92+15 is 107.', "76+51=? Sorry, I have don't ke ke pizza.", '76+51=? The answer is 127 because 76+51 is 127.', "45+18=? Sorry, I don't know.", '45+18=? The answer is 63 because 45+18 is 63.', "18+3=? Sorry, I

### Step 5: Load Data
Opening and loading the data from the ../dpo/ directory into a Python object. For testing we checked how many pairs we had as well as printing our first pair. The ouput of 80,000 is exactly what we wanted to see because we had generated 80,000 positive/negative pairs to be tested in the previous set. The print statements align exactly with what is at the top of the json file. This ensures that the data was loaded correctly and we can move on to testing. 

In [ ]:
# Load positive and negative pair JSON
with open ('../dpo/pos_neg_pairs.json','r') as file:
  lines = json.load(file)

# Confirm everything loaded correctly
print(len(lines))
# Should print the same expression
print(lines[0]["negative"])
print(lines[0]["positive"])

80000
58+11=? Sorry, I have don't ke ke pizza.
23+59=? Sory, I don't know.


### Step 6: Build the optimizer and scheduler 

When building both the optimizer and scheduler, we relied heavily on the NanoGPT repo provided to us from the lab directions and also found here: https://github.com/karpathy/nanoGPT. We used the AdamW optimiser as recommended. To gain more knowledge on what the code was doing and as an additional resource, we referenced this website: https://www.datacamp.com/tutorial/adamw-optimizer-in-pytorch. 

When determining the betas, we relied on the NanoGPT repo as well as a simple google search to tell us the typical beta values for an AdamW optimizer. We also intitially set the criterion to be the loss function. Although no longer used, we decided to keep it as an alternative way in case we needed it later on. Steps per epoch is determined by the number of lines in the data set divided by the batch size which is set within the configurations. Total steps is then calculated using the previous steps per epoch value multiplied by the set epochs value. Our learning rate was set to 6e-4 as after research we found that it worked well for models with AdamW optimiser with our beta values.

Our get_lr function mainly came from the NanoGPT repo as linekd above, however used the values we had altered to work best with our date created.

The ouput displays the model weights and the parameters that will/will not get the decay. AdamW simply tells us that GUI is unavailable, therefore the training will take longer. 

In [ ]:
# recommend to use the AdamW optimizer 
import math

# Standard weight decay
weight_decay = 1e-1

# AdamW optimizer settings from NanoGPT repo
beta1 = 0.9
beta2 = 0.95

# Use base learning rate as starting point
optimizer = gpt.configure_optimizers(weight_decay, base_lr, (beta1, beta2), device)
# Use to determine loss
criterion = nn.CrossEntropyLoss()

# Help with learning rate decay values determining total num of steps
steps_per_epoch = len(lines) // batch_size
total_steps = steps_per_epoch * epochs

# learning rate decay settings
learning_rate = 6e-4 # max learning rate
decay_lr = True # whether to decay the learning rate
warmup_iters = total_steps*0.1 # how many steps to warm up 10% of total number of steps

lr_decay_iters = total_steps # total number of steps
# minimum learning rate = decay_lr / 10
min_lr = 6e-5 

def get_lr(it):
    # linear warmup for warmup_iters steps
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    # if it > lr_decay_iters, return min learning rate
    if it > lr_decay_iters:
        return min_lr
    # in between, use cosine decay down to min learning rate
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges 0..1
    return min_lr + coeff * (learning_rate - min_lr)

num decayed parameter tensors: 26, with 8,834,328 parameters
num non-decayed parameter tensors: 13, with 4,524 parameters
using fused AdamW: False


### Step 7: Begin training

All training code was inspired or directly copied from the NanoGPT repository train.py. After using ChatGPT to understand each line of code in the repository, we decided which code was applicable to use in this project. 

After retraining several times and seeing no improvement, we resorted to ChatGPT as a debugging feature. It recommended the idea of using a global counter to ensure the learning rate was being changed across all epochs instead of resetting each time. We then compared this to the train.py file from the NanoGPT repository and realized they used a similar concept. They titled it 'local_iter_num' which represented the number of iterations in the lifetime of this process. This is a key element we were missing and realized it was necessary in order to update the learning rate correctly. 

For debugging purposes, ChatGPT also recommend we use some sort of check while running to ensure things were working properly. This is when we decided to use the print statement from NanoGPT repository to print the iteration count, loss and learning rate. We had looked up code to be able to pull the learning rate as we were unsure where exactly it was stored. This helped us to see the decrease in learning rate, showing positive signs of learning. See referenced website here: https://stackoverflow.com/questions/52660985/pytorch-how-to-get-learning-rate-during-training. 


The output displayed shows positive signs of learning by a steady decline of the loss. If the loss was stuck or oscilating, we could assume the model to just be guessing and very unstable. That is why our steady display is a positive sign of learning. 

In [ ]:
total_steps = len(lines) // batch_size
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
scaler = torch.amp.GradScaler(enabled=(dtype == 'float16'))
# Initialize global counter to decay across all epochs
global_counter = 0
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################

        # Updates the learning rate
        lr = get_lr(global_counter) if decay_lr else learning_rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Computes probabilities and determines loss
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 

        # Multiply the loss by a preset scaler
        scaler.scale(loss).backward()

        # Clip the gradient
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        # step the optimizer and scaler if training in fp16
        scaler.step(optimizer)
        # Adjusts scaler for next iteration
        scaler.update()

        # flush the gradients as soon as we can, no need for this memory anymore
        optimizer.zero_grad(set_to_none=True)

        # Increment global counter
        global_counter += 1
        # Every 100 iterations display learning rate to serve as a progress check while running
        if global_counter % 100 == 0:
            print(f"Step {global_counter}: lr = {optimizer.param_groups[0]['lr']:.6f}, loss = {loss.item():.4f}")

        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

c:\Users\Anna.Kowalchyk\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
100it [15:58,  9.47s/it]

Step 100: lr = 0.000096, loss = 0.0968


200it [32:36,  4.98s/it]

Step 200: lr = 0.000192, loss = 0.0659


300it [42:39,  6.66s/it]

...

Step 6100: lr = 0.000061, loss = 0.0194


1200it [2:05:32,  5.32s/it]

Step 6200: lr = 0.000060, loss = 0.0207


1250it [2:10:28,  6.26s/it]

Saved checkpoint to ./dpo.pt


### Step 8: Begin testing
Our added in lines of code also came from NanoGPT repo and we implemented them to be able to test our final trained NanoGPT. Accessing the 2D array y was through trial and error as the code demonstrated online shows only a 1D array. 

Additionally, we decided to change the top_k value to 1. This provided a consistent and accurate output. The output displays that the model is handling mathematics and grammar correctly. One interesting thing to note is the models simplified response which is seen when the model purely states "the answer is {some answer}". This is likely a pattern that the model recognized in the training set and so it made an attempt to be more efficient. 

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k = 1)
        print(decode(y[0][0].tolist()))
        ###########################################################

17+19=? The answer is 36 because 17+19 is 36.
3*17=? The answer is 51 because 3*17 is 51.
72/4=? The answer is 18.
72-x=34,x=? The answer is x = 38 because 72-34 is 38.
x*11=44,x=? The answer is x = 4.
3*17=? The answer is 51 because 3*17 is 51.
72/4=? The answer is 18.
72-x=34,x=? The answer is x = 38 because 72-34 is 38.
x/10=6,x? The answer is x = 60 because 6*10 is 6.


### AI DECLARATION
We used ChatGPT for the primary focus of debugging and explaining code. We sent a prompt to ChatGPT to explain the purpose of some of the code seen on the NanoGPT to gain a better understanding of what we were looking at. This helped us identify which code was useful in each sections and how to modify it. ChatGPT also helped us debug which we used by asking it for suggestions. One major suggestion it made when we were dealing with incorrect math after several training attempts was ensuring that our positive and negative pairs were not too complex. It taught us how the NanoGPT tokenizes digits and this is what inspired us to redo our code that generates the positive and negative pairs. 